In [1]:
# Βιβλιοθήκες για τη διαχείριση αρχείων, δεδομένων και API requests
import os
import html
import requests
import pandas as pd

from dotenv import load_dotenv
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import display


# Φόρτωση των μεταβλητών που είναι αποθηκευμένες στο αρχείο .env
load_dotenv()

# Ανάκτηση του YouTube API key χωρίς να εμφανίζεται μέσα στον κώδικα
API_KEY = os.getenv("YOUTUBE_API_KEY")

# Διακοπή της εκτέλεσης αν το API key δεν βρεθεί
if not API_KEY:
    raise ValueError("Δεν βρέθηκε το YOUTUBE_API_KEY στο αρχείο .env")

# Δημιουργία session για την επαναχρησιμοποίηση της σύνδεσης με το API
session = requests.Session()

print("Το API key φορτώθηκε επιτυχώς από το .env.")

Το API key φορτώθηκε επιτυχώς από το .env.


In [2]:
# Τα 4 θέματα και οι αντίστοιχοι όροι αναζήτησης στο YouTube
TOPICS = {
    "Football": "football",
    "Climate Change": "climate change",
    "Video Games": "video games",
    "Artificial Intelligence": "artificial intelligence"
}

# Αριθμός σχολίων που θέλουμε να συλλέξουμε για κάθε θέμα
TARGET_PER_TOPIC = 250

# Μέγιστος αριθμός βίντεο που θα αναζητηθούν για κάθε θέμα
VIDEOS_PER_TOPIC = 40

# Διευθύνσεις του YouTube API για αναζήτηση βίντεο και συλλογή σχολίων
SEARCH_URL = "https://www.googleapis.com/youtube/v3/search"
COMMENTS_URL = "https://www.googleapis.com/youtube/v3/commentThreads"

# Εμφάνιση μιας σύντομης περίληψης των ρυθμίσεων
print(f"Topics: {len(TOPICS)}")
print(f"Στόχος ανά topic: {TARGET_PER_TOPIC}")
print(f"Συνολικός στόχος: {TARGET_PER_TOPIC * len(TOPICS)} σχόλια")

Topics: 4
Στόχος ανά topic: 250
Συνολικός στόχος: 1000 σχόλια


In [3]:
# Λίστα στην οποία θα αποθηκευτούν τα στοιχεία των βίντεο
videos = []

# Αναζήτηση βίντεο ξεχωριστά για κάθε topic
for topic, search_query in TOPICS.items():

    # Παράμετροι που στέλνονται στο YouTube Search API
    params = {
        "part": "snippet",
        "q": search_query,
        "type": "video",
        "maxResults": VIDEOS_PER_TOPIC,
        "order": "relevance",
        "relevanceLanguage": "en",
        "safeSearch": "moderate",
        "key": API_KEY
    }

    # Αποστολή του αιτήματος στο API
    response = session.get(
        SEARCH_URL,
        params=params,
        timeout=30
    )

    # Διακοπή της διαδικασίας αν το API επιστρέψει σφάλμα
    if response.status_code != 200:
        error = response.json().get("error", {})

        raise RuntimeError(
            f"YouTube Search API error {response.status_code}: "
            f"{error.get('message', 'Unknown error')}"
        )

    # Μετατροπή της απάντησης του API σε Python dictionary
    data = response.json()

    # Καταγραφή των βασικών πληροφοριών κάθε βίντεο
    for search_rank, item in enumerate(
        data.get("items", []),
        start=1
    ):
        snippet = item["snippet"]
        video_id = item["id"]["videoId"]

        videos.append({
            "topic": topic,
            "search_query": search_query,
            "search_rank": search_rank,
            "video_id": video_id,
            "video_title": html.unescape(snippet["title"]),
            "video_channel": html.unescape(
                snippet["channelTitle"]
            ),
            "video_published_at": snippet["publishedAt"],
            "video_url": (
                f"https://www.youtube.com/watch?v={video_id}"
            )
        })


# Μετατροπή της λίστας σε DataFrame και αφαίρεση διπλότυπων βίντεο
videos_df = (
    pd.DataFrame(videos)
    .drop_duplicates(subset=["topic", "video_id"])
    .reset_index(drop=True)
)

# Έλεγχος του αριθμού των βίντεο που βρέθηκαν για κάθε topic
video_summary = (
    videos_df
    .groupby("topic")
    .agg(videos_found=("video_id", "nunique"))
    .reindex(TOPICS.keys())
)

display(video_summary)

print(f"Συνολικά videos: {len(videos_df)}")

,videos_found
topic,
Football,37
Climate Change,40
Video Games,40
Artificial Intelligence,40


Συνολικά videos: 157


In [4]:
def get_api_error(response):
    """Επιστρέφει τον λόγο και το μήνυμα ενός API error."""

    try:
        # Ανάκτηση των πληροφοριών του σφάλματος από την απάντηση
        error = response.json().get("error", {})
        errors = error.get("errors", [])

        reason = (
            errors[0].get("reason", "")
            if errors
            else ""
        )

        message = error.get(
            "message",
            "Unknown YouTube API error"
        )

    # Χρησιμοποιείται όταν η απάντηση δεν μπορεί να μετατραπεί σε JSON
    except ValueError:
        reason = ""
        message = response.text

    return reason, message


def fetch_comment_page(video_row, page_token=None):
    """Συλλέγει μία σελίδα από top-level σχόλια ενός βίντεο."""

    # Παράμετροι για τη συλλογή έως 100 σχολίων
    params = {
        "part": "snippet",
        "videoId": video_row["video_id"],
        "maxResults": 100,
        "order": "time",
        "textFormat": "plainText",
        "key": API_KEY
    }

    # Χρησιμοποιείται όταν ζητάμε την επόμενη σελίδα σχολίων
    if page_token:
        params["pageToken"] = page_token

    # Αποστολή αιτήματος στο YouTube Comments API
    response = session.get(
        COMMENTS_URL,
        params=params,
        timeout=30
    )

    # Διαχείριση πιθανών σφαλμάτων του API
    if response.status_code != 200:
        reason, message = get_api_error(response)

        # Περιπτώσεις στις οποίες το βίντεο παραλείπεται
        unavailable_reasons = {
            "commentsDisabled",
            "videoNotFound",
            "forbidden"
        }

        if reason in unavailable_reasons:
            print(
                f"Παράλειψη video {video_row['video_id']}: "
                f"{reason}"
            )
            return [], None

        # Τα υπόλοιπα σφάλματα σταματούν την εκτέλεση
        raise RuntimeError(
            f"YouTube Comments API error "
            f"{response.status_code}: {message}"
        )

    data = response.json()

    # Καταγραφή της ημερομηνίας και ώρας συλλογής
    collected_at = datetime.now(timezone.utc).isoformat()

    records = []

    # Επεξεργασία των σχολίων που επέστρεψε το API
    for item in data.get("items", []):
        thread_snippet = item["snippet"]
        top_comment = thread_snippet["topLevelComment"]
        comment_snippet = top_comment["snippet"]

        comment_id = top_comment["id"]

        # Το author channel ID μπορεί να μην υπάρχει σε όλα τα σχόλια
        author_channel = (
            comment_snippet
            .get("authorChannelId", {})
            .get("value")
        )

        # Προτιμάται το αρχικό κείμενο και χρησιμοποιείται εναλλακτικά το display text
        text = (
            comment_snippet.get("textOriginal")
            or comment_snippet.get("textDisplay", "")
        )

        # Αποθήκευση των διαθέσιμων στοιχείων του σχολίου
        records.append({
            "comment_id": comment_id,
            "author_name": html.unescape(
                comment_snippet.get(
                    "authorDisplayName",
                    ""
                )
            ),
            "author_channel_id": author_channel,
            "topic": video_row["topic"],
            "search_query": video_row["search_query"],
            "text": text,
            "published_at": comment_snippet.get(
                "publishedAt"
            ),
            "updated_at": comment_snippet.get(
                "updatedAt"
            ),
            "like_count": comment_snippet.get(
                "likeCount",
                0
            ),
            "reply_count": thread_snippet.get(
                "totalReplyCount",
                0
            ),
            "video_id": video_row["video_id"],
            "video_title": video_row["video_title"],
            "video_channel": video_row["video_channel"],
            "search_rank": video_row["search_rank"],
            "comment_url": (
                "https://www.youtube.com/watch?"
                f"v={video_row['video_id']}"
                f"&lc={comment_id}"
            ),
            "collected_at": collected_at
        })

    # Επιστρέφονται τα σχόλια και το token της επόμενης σελίδας
    return records, data.get("nextPageToken")

In [5]:
# Εδώ θα αποθηκευτεί το τελικό DataFrame κάθε topic
topic_datasets = []

# Χρησιμοποιείται για την αποφυγή διπλότυπων σχολίων μεταξύ των topics
global_comment_ids = set()

for topic in TOPICS:

    print(f"\nΣυλλογή topic: {topic}")

    # Επιλογή και ταξινόμηση των βίντεο του συγκεκριμένου topic
    topic_videos = (
        videos_df[videos_df["topic"] == topic]
        .sort_values("search_rank")
        .reset_index(drop=True)
    )

    # Προσωρινή αποθήκευση σχολίων και IDs για το τρέχον topic
    topic_pool = []
    topic_comment_ids = set()

    # Αποθήκευση του επόμενου page token για κάθε βίντεο
    next_page_tokens = {}

    # Συλλογή της πρώτης σελίδας σχολίων από κάθε βίντεο
    for _, video_row in topic_videos.iterrows():

        batch, next_token = fetch_comment_page(
            video_row
        )

        # Προσθήκη μόνο μοναδικών σχολίων
        for record in batch:
            comment_id = record["comment_id"]

            if (
                comment_id not in topic_comment_ids
                and comment_id not in global_comment_ids
            ):
                topic_pool.append(record)
                topic_comment_ids.add(comment_id)

        # Αποθήκευση του token αν υπάρχει επόμενη σελίδα
        if next_token:
            next_page_tokens[
                video_row["video_id"]
            ] = next_token

    # Συλλογή επιπλέον σελίδων μέχρι να φτάσουμε τα 250 σχόλια
    while (
        len(topic_pool) < TARGET_PER_TOPIC
        and next_page_tokens
    ):
        comments_before = len(topic_pool)

        for _, video_row in topic_videos.iterrows():
            video_id = video_row["video_id"]

            # Παράλειψη βίντεο που δεν έχει άλλη σελίδα σχολίων
            if video_id not in next_page_tokens:
                continue

            batch, next_token = fetch_comment_page(
                video_row,
                next_page_tokens[video_id]
            )

            # Προσθήκη μόνο σχολίων που δεν έχουν ήδη συλλεχθεί
            for record in batch:
                comment_id = record["comment_id"]

                if (
                    comment_id not in topic_comment_ids
                    and comment_id not in global_comment_ids
                ):
                    topic_pool.append(record)
                    topic_comment_ids.add(comment_id)

            # Ενημέρωση ή αφαίρεση του page token του βίντεο
            if next_token:
                next_page_tokens[video_id] = next_token
            else:
                del next_page_tokens[video_id]

            # Σταματάμε μόλις καλυφθεί ο στόχος
            if len(topic_pool) >= TARGET_PER_TOPIC:
                break

        # Προστασία από συνεχή επανάληψη αν δεν βρεθούν νέα σχόλια
        if len(topic_pool) == comments_before:
            break

    # Διακοπή αν δεν βρέθηκαν αρκετά σχόλια για το topic
    if len(topic_pool) < TARGET_PER_TOPIC:
        raise RuntimeError(
            f"Βρέθηκαν μόνο {len(topic_pool)} σχόλια "
            f"για το topic '{topic}'."
        )

    topic_df = pd.DataFrame(topic_pool)

    # Αρίθμηση των σχολίων μέσα σε κάθε βίντεο
    topic_df["position_in_video"] = (
        topic_df
        .groupby("video_id")
        .cumcount()
    )

    # Επιλογή 250 σχολίων με όσο γίνεται καλύτερη κατανομή μεταξύ των βίντεο
    topic_df = (
        topic_df
        .sort_values(
            [
                "position_in_video",
                "search_rank",
                "published_at"
            ],
            ascending=[True, True, False]
        )
        .head(TARGET_PER_TOPIC)
        .drop(columns="position_in_video")
        .reset_index(drop=True)
    )

    # Καταγραφή των τελικών comment IDs ώστε να μην επαναχρησιμοποιηθούν
    global_comment_ids.update(
        topic_df["comment_id"].tolist()
    )

    topic_datasets.append(topic_df)

    print(
        f"Ολοκληρώθηκε: {len(topic_df)} σχόλια "
        f"από {topic_df['video_id'].nunique()} videos"
    )


# Ένωση των τεσσάρων topics σε ένα ενιαίο DataFrame
comments_df = pd.concat(
    topic_datasets,
    ignore_index=True
)


Συλλογή topic: Football
Παράλειψη video e_-DZUNK6oc: commentsDisabled
Παράλειψη video Zgpw8zwOAyg: commentsDisabled
Παράλειψη video pGHqAH9dDE8: commentsDisabled
Ολοκληρώθηκε: 250 σχόλια από 32 videos

Συλλογή topic: Climate Change
Παράλειψη video SDRxfuEvqGg: commentsDisabled
Παράλειψη video KePzMGhoFjE: commentsDisabled
Παράλειψη video zl-7W_wB_fw: commentsDisabled
Παράλειψη video 7yHcXQoR1zA: commentsDisabled
Ολοκληρώθηκε: 250 σχόλια από 33 videos

Συλλογή topic: Video Games
Ολοκληρώθηκε: 250 σχόλια από 40 videos

Συλλογή topic: Artificial Intelligence
Παράλειψη video _19pRsZRiz4: commentsDisabled
Παράλειψη video JcXKbUIebrU: commentsDisabled
Παράλειψη video qD-o2lDQwa8: commentsDisabled
Παράλειψη video oBUAQGwzGk0: commentsDisabled
Παράλειψη video ttIOdAdQaUE: commentsDisabled
Ολοκληρώθηκε: 250 σχόλια από 35 videos


In [6]:
# Τελικές στήλες και σειρά εμφάνισής τους στο dataset
FINAL_COLUMNS = [
    "comment_id",
    "author_name",
    "author_channel_id",
    "topic",
    "search_query",
    "text",
    "published_at",
    "updated_at",
    "like_count",
    "reply_count",
    "video_id",
    "video_title",
    "video_channel",
    "search_rank",
    "comment_url",
    "collected_at"
]

comments_df = comments_df[FINAL_COLUMNS]

# Περίληψη των σχολίων και των βίντεο ανά topic
summary = (
    comments_df
    .groupby("topic")
    .agg(
        comments=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        videos_used=("video_id", "nunique")
    )
    .reindex(TOPICS.keys())
)

display(summary)

# Έλεγχος για διπλότυπα comment IDs
duplicate_ids = comments_df["comment_id"].duplicated().sum()

print(f"Συνολικά σχόλια: {len(comments_df)}")
print(f"Διπλότυπα comment IDs: {duplicate_ids}")

# Αυτόματοι έλεγχοι για την επιβεβαίωση του τελικού dataset
assert len(comments_df) == 1000
assert comments_df["comment_id"].is_unique
assert (
    comments_df.groupby("topic")
    .size()
    .eq(TARGET_PER_TOPIC)
    .all()
)

# Δημιουργία του φακέλου data/raw 
output_folder = Path("../data/raw")
output_folder.mkdir(parents=True, exist_ok=True)

# Όνομα και διαδρομή του τελικού raw αρχείου
output_file = output_folder / "youtube_comments_raw.csv"

# Αποθήκευση σε CSV 
comments_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nΑποθηκεύτηκε επιτυχώς στο: {output_file}")

,comments,unique_comments,videos_used
topic,,,
Football,250,250,32
Climate Change,250,250,33
Video Games,250,250,40
Artificial Intelligence,250,250,35


Συνολικά σχόλια: 1000
Διπλότυπα comment IDs: 0

Αποθηκεύτηκε επιτυχώς στο: ../data/raw/youtube_comments_raw.csv


In [7]:
# Εμφάνιση 20 ενδεικτικών σχολίων από κάθε topic

SAMPLE_PER_TOPIC = 20

SAMPLE_COLUMNS = [
    "comment_id",
    "author_name",
    "author_channel_id",
    "topic",
    "search_query",
    "text",
    "published_at",
    "updated_at",
    "like_count",
    "reply_count",
    "video_id",
    "video_title",
    "video_channel",
    "search_rank",
    "comment_url",
    "collected_at"
]

for topic in TOPICS:

    topic_sample = (
        comments_df[
            comments_df["topic"] == topic
        ][SAMPLE_COLUMNS]
        .head(SAMPLE_PER_TOPIC)
        .reset_index(drop=True)
    )

    print(f"\n{topic}: {len(topic_sample)} ενδεικτικά σχόλια")

    with pd.option_context(
        "display.max_colwidth",
        200
    ):
        display(topic_sample)


Football: 20 ενδεικτικά σχόλια


,comment_id,author_name,author_channel_id,topic,search_query,text,published_at,updated_at,like_count,reply_count,video_id,video_title,video_channel,search_rank,comment_url,collected_at
0,UgwCR9UDCidpwsey5Z54AaABAg,@hayleybrown6954,UCqurhWqNrj1WuUa1ZXGIzhA,Football,football,7:09 kid in the front plays on my team,2026-08-08T07:52:06Z,2026-08-08T07:52:06Z,0,0,OjcYwDUBcq0,Football Moments You Missed On TV,Chuff Footy Reacts,2,https://www.youtube.com/watch?v=OjcYwDUBcq0&lc=UgwCR9UDCidpwsey5Z54AaABAg,2026-08-08T22:08:08.064756+00:00
1,UgyR2PQD_ZodUbZ-4l54AaABAg,@NinetyVirus,UC9aSRBH_gEtPD5E7cWO10Dw,Football,football,"Greatest game on earth, but also the scariest? 🥶",2026-05-08T10:35:10Z,2026-05-08T10:35:10Z,115,12,9VnSsqGdhZA,Scariest Moments In Football,Ninety,3,https://www.youtube.com/watch?v=9VnSsqGdhZA&lc=UgyR2PQD_ZodUbZ-4l54AaABAg,2026-08-08T22:08:08.371566+00:00
2,UgxbHPmE2e23uXxSpcV4AaABAg,@avaaesthic,UClA-rMB5n-xNyHCcAHV6dOg,Football,football,I support liverpool,2026-08-03T08:40:46Z,2026-08-03T08:40:46Z,0,0,G7_4h1cCIsE,GUESS THE CLUB BY PLAYERS' HAIR | FOOTBALL QUIZ 2026,The Best Football Quiz,4,https://www.youtube.com/watch?v=G7_4h1cCIsE&lc=UgxbHPmE2e23uXxSpcV4AaABAg,2026-08-08T22:08:08.548432+00:00
3,UgxLBvsU6eZhWhcYA1x4AaABAg,@TheWingroveFamily,UCyY4jVInowKG9Nm20xXGHFw,Football,football,🚨YES GUYS! Who's team is better? Team Billy & Dustie or Team Roman & Amelie? Let us know!!⬇,2026-07-10T15:08:13Z,2026-07-10T15:08:13Z,122,73,qHIiLIemo6k,INSANE WORLD CUP FC26 CARD BATTLE!!,The Wingrove Family,5,https://www.youtube.com/watch?v=qHIiLIemo6k&lc=UgxLBvsU6eZhWhcYA1x4AaABAg,2026-08-08T22:08:08.995770+00:00
4,UgwoQdncz-hGBpPf5ix4AaABAg,@QuestionKid,UCD8koChmOzEPK3uuiCe8QLg,Football,football,W Nick and chuff,2026-08-07T08:57:42Z,2026-08-07T08:57:42Z,0,0,IqcwXwCC9Fc,Best POWER Goals In Football,Chuff Footy Reacts,6,https://www.youtube.com/watch?v=IqcwXwCC9Fc&lc=UgwoQdncz-hGBpPf5ix4AaABAg,2026-08-08T22:08:09.229685+00:00
5,UgzkYdjfSBCmg3p0JfB4AaABAg,@NinetyVirus,UC9aSRBH_gEtPD5E7cWO10Dw,Football,football,"Trust me, even I can score that 1CM goal 😏",2026-02-05T11:30:23Z,2026-02-05T11:30:23Z,334,14,AqrQzFRJ02U,Football Goals But The Distance KEEPS Increasing,Ninety,7,https://www.youtube.com/watch?v=AqrQzFRJ02U&lc=UgzkYdjfSBCmg3p0JfB4AaABAg,2026-08-08T22:08:09.448433+00:00
6,UgxR3IPGYwCcbCVc-654AaABAg,@JacksonJunior-n5c,UC00r_-dyfKGAGpJwEFWKhaQ,Football,football,Well edit,2026-08-08T17:59:10Z,2026-08-08T17:59:10Z,0,0,D_oSbDhy5F4,Ronaldo Taught Yamal a Football Lesson 🥶☠️,Flush,8,https://www.youtube.com/watch?v=D_oSbDhy5F4&lc=UgxR3IPGYwCcbCVc-654AaABAg,2026-08-08T22:08:09.817347+00:00
7,UgzI1YKHnlpVF7VvsHJ4AaABAg,@ဝိရောဓိ,UCdiEijLVy2JLWSoHp8m9b3g,Football,football,Don't cry Laotians. \nMy country's dictator told them that they will be jailed if there is no win.,2026-08-08T08:45:14Z,2026-08-08T08:45:14Z,0,0,KMI5C_rSqUI,Myanmar 7-2 Laos | ASEAN Hyundai Cup 2026 | Match Highlights,ASEAN United FC,10,https://www.youtube.com/watch?v=KMI5C_rSqUI&lc=UgzI1YKHnlpVF7VvsHJ4AaABAg,2026-08-08T22:08:10.109957+00:00
8,UgyVn7zeFLo--AJcyF94AaABAg,@Waingro27,UCwGe-nvJzllPM9xIc5HyTjw,Football,football,Are you in england,2026-08-08T21:04:58Z,2026-08-08T21:04:58Z,0,0,_8ggEN2ODqQ,"Every Time I Score, The Football CHANGES!",Chuffsters,11,https://www.youtube.com/watch?v=_8ggEN2ODqQ&lc=UgyVn7zeFLo--AJcyF94AaABAg,2026-08-08T22:08:10.326592+00:00
9,UgxqqkfPgCN3GIV-mjd4AaABAg,@FAIZCAPE_EDITZ,UCIyN4n0whFMA80IztppW9DQ,Football,football,Bro can I use your clips for edit ???,2026-08-08T17:44:45Z,2026-08-08T17:44:45Z,0,0,02h7MB4Re4o,Best Football Skills 2026,SportsHD,12,https://www.youtube.com/watch?v=02h7MB4Re4o&lc=UgxqqkfPgCN3GIV-mjd4AaABAg,2026-08-08T22:08:10.470813+00:00



Climate Change: 20 ενδεικτικά σχόλια


,comment_id,author_name,author_channel_id,topic,search_query,text,published_at,updated_at,like_count,reply_count,video_id,video_title,video_channel,search_rank,comment_url,collected_at
0,UgxmSjOhV2UFkkXuPRB4AaABAg,@beldurnik21,UCxTHlnEYPXDByIq8wH7783g,Climate Change,climate change,Shill,2026-08-08T22:05:24Z,2026-08-08T22:05:24Z,0,0,3nlU3cs8Yhs,Count Binface has climate policies?!,Simon Clark,1,https://www.youtube.com/watch?v=3nlU3cs8Yhs&lc=UgxmSjOhV2UFkkXuPRB4AaABAg,2026-08-08T22:08:16.121636+00:00
1,Ugw8gsKrkjFxF61LjUR4AaABAg,@Timorias,UCPW-_4k8bWibmlqINpBOj-w,Climate Change,climate change,"Some context: 2026 isn’t breaking the records JUST set in the last 5 years. (Including when we went on a streak, breaking global average temp records for 13 months straight) If we didn’t count the...",2026-08-08T06:05:28Z,2026-08-08T06:05:28Z,0,0,xBGSlxv38nI,2026 isn't breaking heat records - yet. El Nino is forecast to spike temperatures,Associated Press,3,https://www.youtube.com/watch?v=xBGSlxv38nI&lc=Ugw8gsKrkjFxF61LjUR4AaABAg,2026-08-08T22:08:16.415207+00:00
2,UgwuVliL07dfo1a9YOd4AaABAg,@traceyjones6641,UCcnrwRqCbUzISLvH2Ir53Og,Climate Change,climate change,It all matters,2026-08-08T21:30:45Z,2026-08-08T21:30:45Z,0,0,FAfDR5u5QLY,Dr. Mustafa Santiago Ali: Climate Change Is Raising Food Prices And Hurting Black Communities,Roland S. Martin,4,https://www.youtube.com/watch?v=FAfDR5u5QLY&lc=UgwuVliL07dfo1a9YOd4AaABAg,2026-08-08T22:08:16.515125+00:00
3,UgzvnUPEvioHdTc5Z7t4AaABAg,@amiava4213,UCG6A7AW8wew3QdhAo3PVP3A,Climate Change,climate change,"Isn’t this glorious? The climate house of cards is finally collapsing.\n\nThe prevailing climate change narrative took a big hit in recent days, as scientists who comprise the United Nations’ Int...",2026-06-03T11:17:14Z,2026-06-03T11:18:07Z,1,2,vM6SRQqyvWc,Climate change is fueling more extreme weather,Associated Press,5,https://www.youtube.com/watch?v=vM6SRQqyvWc&lc=UgzvnUPEvioHdTc5Z7t4AaABAg,2026-08-08T22:08:16.667684+00:00
4,UgyvYk15M60ABhEJkjp4AaABAg,@chrishughes7627,UCLKMsPxDYpLnBoG1aAxkVCg,Climate Change,climate change,"Laura,tell the real reason our temperatures are increasing. Clear skies and lack of pollution, nothing to reflect the sun back into space,hence the sun is now like a super powered led light which ...",2026-07-24T06:35:36Z,2026-07-24T06:35:36Z,0,0,nwzIpxPsy2E,Record heatwave 'impossible' without climate change #weather #heatwave #climatechange,ITV News,6,https://www.youtube.com/watch?v=nwzIpxPsy2E&lc=UgyvYk15M60ABhEJkjp4AaABAg,2026-08-08T22:08:16.933907+00:00
5,UgwAYiw1izwrlDac8vF4AaABAg,@moxiesaint-clare4257,UCb1b1kpmrN10bAMGuf8z1GQ,Climate Change,climate change,"I remember building a snowman in the back garden 1978-1985 , stayed there for weeks. We had snow every winter but it got less and less each year. Summers were warm 18-23 c wasn't until 97, 98 the...",2026-07-11T04:01:06Z,2026-07-11T04:01:06Z,0,0,DOPcMMZ6FHU,1976 was an outlier. This isn't. ☀️ #summer #sun #climatechange #physics #nature,The Royal Society,7,https://www.youtube.com/watch?v=DOPcMMZ6FHU&lc=UgwAYiw1izwrlDac8vF4AaABAg,2026-08-08T22:08:17.049079+00:00
6,Ugw7GQnitlXXaJXTghx4AaABAg,@AddiK-k5d,UCv91zphNFBjkDVl_rIIsqgQ,Climate Change,climate change,This is bad for the environment,2026-07-25T23:23:11Z,2026-07-25T23:23:11Z,0,0,OGkTjqsqU5c,"Global sea levels could rise more than expected because of climate change, according to a new study",NPR,8,https://www.youtube.com/watch?v=OGkTjqsqU5c&lc=Ugw7GQnitlXXaJXTghx4AaABAg,2026-08-08T22:08:17.285999+00:00
7,UgzOtRAzUE0fclLHNwJ4AaABAg,@jimdellavecchia4594,UC0hOPFzX3NAQK2FlvkoYZTQ,Climate Change,climate change,I'm 59 and remember when we were all going to die because the ozone layer was going to disappear,2026-08-08T21:30:44Z,2026-08-08T21:30:44Z,0,0,D9yizR2fab8,Where Have All the Climate Alarmists Gone? | 5-Minute Videos | PragerU,PragerU,9,https://www.youtube.com/watch?v=D9yizR2fab8&lc=UgzOtRAzUE0fclLHNwJ4AaABAg,2026-08-08T22:08:17.539437+00:00
8,Ugyiqa2cRJsDBWihMuZ4AaAB


Video Games: 20 ενδεικτικά σχόλια


,comment_id,author_name,author_channel_id,topic,search_query,text,published_at,updated_at,like_count,reply_count,video_id,video_title,video_channel,search_rank,comment_url,collected_at
0,UgwAa9k5_gzNbuklkjJ4AaABAg,@asukashyap19,UCk1BDZJS3vEFjl_7Cr03viA,Video Games,video games,"Im writing this for my man (since my phone is broken so im commenting from other account) \n\n\n\nHeaven is a place on earth with you , ill dedicate this song for you my love , as long as im alive...",2026-08-08T12:49:11Z,2026-08-08T12:49:11Z,0,0,cE6wxDqdOV0,Lana Del Rey - Video Games,LanaDelReyVEVO,1,https://www.youtube.com/watch?v=cE6wxDqdOV0&lc=UgwAa9k5_gzNbuklkjJ4AaABAg,2026-08-08T22:08:23.384764+00:00
1,UgyHsKLAgy_Di6bl69N4AaABAg,@rart31,UCqcV729q0wxAAS68RyNjGbg,Video Games,video games,The scout nice,2026-08-08T20:31:28Z,2026-08-08T20:31:28Z,0,0,GNJtPFXUnm4,Tenacious D - Video Games,Tenacious D,2,https://www.youtube.com/watch?v=GNJtPFXUnm4&lc=UgyHsKLAgy_Di6bl69N4AaABAg,2026-08-08T22:08:23.586904+00:00
2,Ugy4HHRBDe3svd4Hf4Z4AaABAg,@KianmehrEyvazi,UCnPnvVQbt5lgFddTdww3xAQ,Video Games,video games,Minecraft,2026-08-07T06:09:49Z,2026-08-07T06:09:49Z,0,0,8kIUR-TA33Y,Best Video Games of All Time,Sambucha,3,https://www.youtube.com/watch?v=8kIUR-TA33Y&lc=Ugy4HHRBDe3svd4Hf4Z4AaABAg,2026-08-08T22:08:23.844706+00:00
3,UgwY8r5mLgeeWAvWDih4AaABAg,@tinamuller9638,UCXR_1KoQGHt1cmEB1kOAWgQ,Video Games,video games,The best Song she has!,2026-08-08T21:15:54Z,2026-08-08T21:15:54Z,0,0,2r3qSKV8TZw,"Lana Del Rey Performs ""Video Games"" | Letterman",Letterman,4,https://www.youtube.com/watch?v=2r3qSKV8TZw&lc=UgwY8r5mLgeeWAvWDih4AaABAg,2026-08-08T22:08:24.103144+00:00
4,Ugzi13qs19D3vFKAETJ4AaABAg,@shrewdfc,UCG6NJ_1BWQEcaAWMo2XDGvg,Video Games,video games,🥶💙,2026-07-29T09:09:13Z,2026-07-29T09:09:13Z,0,0,P3YJcpDZ3fA,Lana Del Rey - Video Games (Glastonbury 2014),BBC Music,5,https://www.youtube.com/watch?v=P3YJcpDZ3fA&lc=Ugzi13qs19D3vFKAETJ4AaABAg,2026-08-08T22:08:24.301824+00:00
5,UgyVoEYa6aramOW5hRB4AaABAg,@singkingkaraoke,UCwTRjvjVge51X-ILJ4i22ew,Video Games,video games,"Thousands of karaoke tracks, unlimited playback! 🎤 Be Swift. Have Styles. Go Grande. Head West! Sing all your favourite artist’s latest tracks on the brand new free Sing King app. https://singking...",2021-03-04T13:31:13Z,2021-03-04T13:31:13Z,224,3,AOVyQC67Q4g,Lana Del Rey - Video Games (Karaoke Version),Sing King,6,https://www.youtube.com/watch?v=AOVyQC67Q4g&lc=UgyVoEYa6aramOW5hRB4AaABAg,2026-08-08T22:08:24.504276+00:00
6,UgxXHKJZ9HLXl9j8a5N4AaABAg,@TDBRICKS,UCUU3GdGuQshZFRGnxAPBf_w,Video Games,video games,0:15 Mario\n1:29 Fortnite\n2:39 Roblox\n2:54 Portal\n3:27 Temple Run\n4:10 Subway Surfers\n4:35 Fruit Ninja\n4:55 Pokemon\n5:15 Among Us\n5:56 Angry Birds\n6:42 Wii Sports\n7:30 Call of Duty\n8:10...,2023-09-18T13:16:23Z,2023-09-18T13:16:23Z,3319,936,2oKoNi7uRgc,POPULAR VIDEO GAMES in LEGO...,TD BRICKS,7,https://www.youtube.com/watch?v=2oKoNi7uRgc&lc=UgxXHKJZ9HLXl9j8a5N4AaABAg,2026-08-08T22:08:24.768768+00:00
7,UgxXRWIqI2TbCW3Ey-p4AaABAg,@LivingtonJoni,UCEAuh75p5c2dYvBIakGMBrQ,Video Games,video games,😂😂😂😂😂😂😂😂is he playing roblox?😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂,2026-08-08T01:17:23Z,2026-08-08T01:17:23Z,0,0,A0NfOmnsWMI,VIDEO GAMES AT 3AM 😝😱 #humor #homealone #babyduck,Packy Films,8,https://www.youtube.com/watch?v=A0NfOmnsWMI&lc=UgxXRWIqI2TbCW3Ey-p4AaABAg,2026-08-08T22:08:24.969154+00:00
8,UgybQsBQwGr8ff52QBd4AaABAg,@ToneFrance,UCu0C2YqRHjyG-5KwozeyLFA,Video Games,video games,Don’t touch my video games is now available on all music streaming platforms!,2025-02-25T15:39:04Z,2025-02-25T15:39:04Z,137,31,eMdhc5pMvFc,Video Games! - (Official Music Video) | ToneFrance Music,ToneFrance,9,https://www.youtube.com/watch?v=eMdhc5pMvFc&lc=UgybQsBQwGr8ff52QBd4AaABAg,2026-08-08T22:08:25.171355+00:00
9,UgzcKp1p95bmW1H-VxV4AaABAg,@KeeFic,UCmIFBj1kKb4eYQZVOI1ZoyQ,Video Games,video games,I love Fortnite🎉🎉🎉🎉,2026-08-08T21:57:59Z,2026-08-08T21:57:59Z,0,0,GhAlJjGkRws,KIDS CRASHING OUT OVER VIDEO GAMES!,Foltyn Reacts,10,https://www.youtube.com/w


Artificial Intelligence: 20 ενδεικτικά σχόλια


,comment_id,author_name,author_channel_id,topic,search_query,text,published_at,updated_at,like_count,reply_count,video_id,video_title,video_channel,search_rank,comment_url,collected_at
0,UgxJ9Bet_FFjoa64d9F4AaABAg,@the_takatika,UC9tIVSYSBNqn2GEv_Rf2MLg,Artificial Intelligence,artificial intelligence,alr can we be fr for a sec. the song is actualy good,2026-08-07T20:08:01Z,2026-08-07T20:08:01Z,0,0,F-pMp8AaXvw,artificial intelligence(SLOWED),JG,1,https://www.youtube.com/watch?v=F-pMp8AaXvw&lc=UgxJ9Bet_FFjoa64d9F4AaABAg,2026-08-08T22:08:32.489829+00:00
1,Ugyg1r2EMd4UW2aTyP14AaABAg,@davidr5515,UCt5Xg_RBK16DOrQf7fTLFtg,Artificial Intelligence,artificial intelligence,Science Built Technologies is developing an AI ecosystem around several departures from conventional AI- A fundamentally different dataset architecture-A Solomonoff-inspired approach to hypothesis...,2026-08-08T15:38:33Z,2026-08-08T15:39:48Z,0,0,siHhK75gf60,Godfather Of AI: We Should Prepare For What's Coming In 2030,Neural Nutshell,2,https://www.youtube.com/watch?v=siHhK75gf60&lc=Ugyg1r2EMd4UW2aTyP14AaABAg,2026-08-08T22:08:32.683810+00:00
2,UgwNlF1Du2raVtCnNxB4AaABAg,@FutureBusinessTech,UCGBO6EahCqQSyXIws1MQdDg,Artificial Intelligence,artificial intelligence,Feel free to like and subscribe if you enjoyed the video. Watch this next video about the Technological Singularity: https://youtu.be/yHEnKwSUzAE.,2023-10-28T15:56:47Z,2023-10-28T15:56:47Z,129,26,tFx_UNW9I1U,The 10 Stages of Artificial Intelligence,Future Business Tech,4,https://www.youtube.com/watch?v=tFx_UNW9I1U&lc=UgwNlF1Du2raVtCnNxB4AaABAg,2026-08-08T22:08:33.016175+00:00
3,UgxFxsWzL0pdTIh2Kqd4AaABAg,@NathanJayMusic,UCvz0eZy-kU-34YnBi00fSDw,Artificial Intelligence,artificial intelligence,"The most amazing thing about this is somebody uses Copilot. I once asked it to explain BitCoin halving to me, it told me all the software I had installed and additional hardware installed. I ask...",2026-08-08T15:49:57Z,2026-08-08T15:49:57Z,0,0,6F8F1K4Eahs,Why are AI agents hacking other companies and have they gone rogue? | BBC Newscast,BBC News,5,https://www.youtube.com/watch?v=6F8F1K4Eahs&lc=UgxFxsWzL0pdTIh2Kqd4AaABAg,2026-08-08T22:08:33.159246+00:00
4,UgwRrws1r2evYsc9zV14AaABAg,@SimplilearnOfficial,UCsvqVGtbbyHaMoevxPAq9Fg,Artificial Intelligence,artificial intelligence,"""🔥Michigan Engineering Professional Certificate in AI and Machine Learning - https://www.simplilearn.com/professional-aiml-program?utm_campaign=ad79nYk2keg&utm_medium=Comments&utm_source=Youtube\n...",2021-09-08T12:54:19Z,2026-07-29T05:45:40Z,214,8,ad79nYk2keg,What Is AI? | Artificial Intelligence | What is Artificial Intelligence? | AI In 5 Mins |Simplilearn,Simplilearn,6,https://www.youtube.com/watch?v=ad79nYk2keg&lc=UgwRrws1r2evYsc9zV14AaABAg,2026-08-08T22:08:33.374850+00:00
5,Ugzcr5Ae4IkGyRno4_t4AaABAg,@LadifewaxNamitey,UCQvDUeHUZConuYCMyYk8sbQ,Artificial Intelligence,artificial intelligence,"Instant subscriber, your content is outstanding!",2025-12-05T21:48:33Z,2025-12-05T21:48:33Z,75,4,m8o2GrbR3d8,SIMPLEST Explanation of How Artificial Intelligence Works? No Jargon | What is AI? How AI works?,Science Simplified 4 All,7,https://www.youtube.com/watch?v=m8o2GrbR3d8&lc=Ugzcr5Ae4IkGyRno4_t4AaABAg,2026-08-08T22:08:33.573681+00:00
6,UgwmwluxG79hEAtzTZR4AaABAg,@ayamdash6427,UCjMCIeylVZdnbY0sucQWxTA,Artificial Intelligence,artificial intelligence,Ayam own new theorem and model\nA good way to create something genuinely novel is to propose a research architecture rather than claim a new state-of-the-art model. Here's a conceptual design.\n\n...,2026-08-07T13:19:28Z,2026-08-07T13:19:28Z,1,0,qYNweeDHiyU,"AI, Machine Learning, Deep Learning and Generative AI Explained",IBM Technology,8,https://www.youtube.com/watch?v=qYNweeDHiyU&lc=UgwmwluxG79hEAtzTZR4AaABAg,2026-08-08T22:08:33.794844+00:00
7,Ugw5IyCxOlVoGqbBZkd4AaABAg,@SimplilearnOfficial,UCsvqVGtbbyHaMoevxPAq9Fg,Artificial Intelligence,artificial intelligence,"""🔥Michigan Engineering Professional Certificate in